# overhead

> Separating fixed per-request overhead from real per-page processing time,
> by measuring at several page counts instead of just one.

A single `predict_time_s` reading, at one page count, cannot tell us how much
of that time is the engine actually working versus fixed cost that happens
on every request regardless of size -- queue admission, authentication,
network round-trips, and, for a polled engine like Datalab, up to
`POLL_INTERVAL_S` of polling-granularity latency (see `estravon-backend`'s
`docs/API.md`, "Honest abstraction edges"). That fixed cost behaves like a
constant added to every measurement of a given engine -- closer to a
systematic bias than to random noise -- so no amount of repeating the same
single-page request will separate it from real work. Separating them needs
requests at *different* page counts.

The model we fit is deliberately simple: request time as a straight line in
page count,

```
time(n) = overhead + per_page_rate * n
```

`overhead` (the intercept) is the fixed cost; `per_page_rate` (the slope) is
the actually-comparable "how fast is this engine at the real work" number
-- the one that should go head-to-head against Replicate's self-reported
GPU-seconds. We get both by measuring `time(n)` at a handful of page
counts and fitting a line through the points, rather than by any single
measurement no matter how carefully taken.

In [ ]:
#| default_exp overhead

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import statistics
import time
from dataclasses import dataclass, field

from estravon_bench.client import Client, LocalEngineProcess

## Repeats at each page count, before fitting anything

Before we can trust a single `time(n)` value enough to put it into the line
fit, we want to know how noisy it is. We measure each page count several
times (`reps`) and keep the **median** as the representative value -- median
rather than mean because request latency is usually right-skewed (an
occasional slow response from queueing or network jitter is far more likely
than an occasional *fast* one), so a mean gets pulled upward by exactly the
kind of outlier we don't want dominating the fit. We also keep the mean and
standard deviation alongside it: a **small** standard deviation at a given
page count means the overhead really is behaving like a fixed constant, as
we'd expect from a systematic cost; a **large** one means something at that
page count is noisy or load-dependent, which is itself worth knowing before
trusting the fitted line.

`PageCountSample` holds the raw and summarised timings for one page count;
`OverheadEstimate` holds the whole calibration -- every `PageCountSample`
plus the fitted `overhead_s` and `per_page_s` and an `r_squared` goodness-of-fit
number (close to 1 means the straight-line model is a good description of
what we measured; well below 1 is a sign the relationship isn't actually
linear over the range tested -- for example if a service starts internally
parallelising above some page count, which would itself be a real finding,
not just measurement noise to explain away).

In [ ]:
#| export
@dataclass
class PageCountSample:
    """Repeated timings at one page count, for one engine."""

    n: int
    times_s: list[float]

    @property
    def median_s(self) -> float:
        return statistics.median(self.times_s)

    @property
    def mean_s(self) -> float:
        return statistics.mean(self.times_s)

    @property
    def stdev_s(self) -> float:
        return statistics.stdev(self.times_s) if len(self.times_s) > 1 else 0.0


@dataclass
class OverheadEstimate:
    """A fitted time(n) = overhead_s + per_page_s * n model for one engine."""

    engine: str
    samples: list[PageCountSample] = field(default_factory=list)
    overhead_s: float = 0.0
    per_page_s: float = 0.0
    r_squared: float = 0.0

    def to_markdown_table(self) -> str:
        header = "| pages | median (s) | mean (s) | stdev (s) | n reps |\n"
        header += "|---|---|---|---|---|\n"
        rows = [
            f"| {s.n} | {s.median_s:.2f} | {s.mean_s:.2f} | {s.stdev_s:.2f} | {len(s.times_s)} |"
            for s in self.samples
        ]
        footer = (
            f"\n\nFitted: `time(n) = {self.overhead_s:.2f}s + {self.per_page_s:.3f}s x n` "
            f"(R² = {self.r_squared:.3f})"
        )
        return header + "\n".join(rows) + footer

## Fitting the line: `_fit_line()`

Ordinary least-squares on `(n, median_s)` pairs -- the standard formula for
the best-fit line through a set of points, implemented directly rather than
pulled in from a numerics library, since two numbers (slope, intercept) and
an R² don't justify a new dependency. `overhead_s` is the intercept
(page count 0, i.e. the part of the time that isn't about pages at all);
`per_page_s` is the slope.

In [ ]:
#| export
def _fit_line(points: list[tuple[float, float]]) -> tuple[float, float, float]:
    """Least-squares fit of y = a + b*x. Returns (a, b, r_squared)."""
    n = len(points)
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    x_bar = sum(xs) / n
    y_bar = sum(ys) / n
    ss_xx = sum((x - x_bar) ** 2 for x in xs)
    ss_xy = sum((x - x_bar) * (y - y_bar) for x, y in points)
    b = ss_xy / ss_xx if ss_xx else 0.0
    a = y_bar - b * x_bar
    ss_tot = sum((y - y_bar) ** 2 for y in ys)
    ss_res = sum((y - (a + b * x)) ** 2 for x, y in points)
    r_squared = 1 - ss_res / ss_tot if ss_tot else 1.0
    return a, b, r_squared

## Running the calibration: `estimate_overhead()`

Same Mode A / Mode B split as `compare()` (`03_compare`), because the same
question applies here too: do we want this package to start a local
instance for us (`engine=`), or point it at one already running
(`engine_url=`)? Unlike `compare()`, this only ever talks to **one** engine
at a time -- the point is to characterise that one engine's overhead, not to
compare engines against each other -- so we start (or reuse) a single
`Client` once and make every request through it, rather than spinning up a
fresh instance per request.

For each page count in `page_counts`, we submit `reps` separate requests
covering pages `1..n` of the given PDF, timing each one from just before
`submit_and_wait()` to just after `fetch_markdown()` returns -- the full
request-to-delivered-text span a real caller experiences, not just the
`predict_time_s` the engine itself reports (which, as the docs note, is
already a narrower, engine-specific measurement). Once every page count has
its `reps` timings, we fit the line through the medians and return the whole
thing as an `OverheadEstimate`.

A single rep failing (a transient outage on the engine's side, unrelated to
this code) is retried, not immediately given up on -- up to `max_retries`
times, with a short backoff between attempts. We keep this **bounded**, not
unlimited: retrying forever would let a genuinely sustained outage hang the
calibration indefinitely instead of finishing with whatever data it could
collect. Only after every attempt at a rep fails do we skip it, and only
give up on a page count entirely if every rep at it failed, since a median
with zero data points isn't a number. Raise `max_retries` (default 2, i.e.
up to 3 attempts) for a known-persistent issue where more patience is
worthwhile.

Every attempt, retry, and give-up prints a line as it happens -- this can
run for several minutes against a slow or unreliable engine, and a
calibration that produces no output until it either finishes or fails is a
worse experience than one that shows its work.

This is meant to be run occasionally as a calibration, not on every
comparison -- it makes `len(page_counts) * reps` real requests (more, if any
needed a retry), which costs real time and (for paid engines) real money.

In [ ]:
#| export
def estimate_overhead(
    pdf_path: str,
    page_counts: list[int] = [1, 5, 10, 20],
    reps: int = 3,
    engine: str | None = None,
    engine_url: str | None = None,
    section_name: str = "overhead_calibration",
    mode: str = "balanced",
    api_key: str | None = None,
    max_wait_s: float = 300.0,
    ready_timeout_s: float = 60.0,
    port: int = 7869,
    estravon_cmd: str = "estravon",
    max_retries: int = 2,
    retry_backoff_s: float = 3.0,
) -> OverheadEstimate:
    """Measure request-to-delivered-text time at several page counts for ONE
    engine, and fit overhead (intercept) vs. per-page rate (slope) through
    the medians. Exactly one of `engine` (spawn a local instance) or
    `engine_url` (use an already-running one) must be given.

    A failed attempt is retried up to `max_retries` times (with
    `retry_backoff_s * (attempt + 1)` seconds between attempts) before that
    rep is given up on -- bounded, not unlimited, so a sustained outage
    can't hang the calibration forever; raise `max_retries` for more
    persistence against a known-transient issue."""
    if (engine is None) == (engine_url is None):
        raise ValueError("pass exactly one of engine= or engine_url=")

    def _run(client: Client) -> OverheadEstimate:
        samples = []
        for n in page_counts:
            times: list[float] = []
            for rep in range(reps):
                elapsed: float | None = None
                for attempt in range(max_retries + 1):
                    print(f"  n={n} rep={rep + 1}/{reps} attempt={attempt + 1}/{max_retries + 1}: submitting...")
                    t0 = time.monotonic()
                    try:
                        result = client.submit_and_wait(
                            pdf_path, section_name, f"1-{n}", mode=mode, max_wait_s=max_wait_s,
                        )
                        if result.get("status") != "done":
                            raise RuntimeError(result.get("error") or f"status={result.get('status')}")
                        client.fetch_markdown(result)
                        elapsed = time.monotonic() - t0
                        print(f"  n={n} rep={rep + 1}/{reps}: done in {elapsed:.2f}s")
                        break
                    except Exception as exc:
                        if attempt < max_retries:
                            backoff = retry_backoff_s * (attempt + 1)
                            print(f"  n={n} rep={rep + 1}/{reps} attempt={attempt + 1}: "
                                  f"failed ({exc}) -- retrying in {backoff:.0f}s")
                            time.sleep(backoff)
                        else:
                            print(f"  n={n} rep={rep + 1}/{reps}: gave up after "
                                  f"{max_retries + 1} attempts ({exc})")
                if elapsed is not None:
                    times.append(elapsed)
            if not times:
                raise RuntimeError(f"all {reps} reps failed at n={n} -- no timing data collected")
            print(f"n={n}: collected {len(times)}/{reps} reps, median={statistics.median(times):.2f}s")
            samples.append(PageCountSample(n=n, times_s=times))
        a, b, r2 = _fit_line([(s.n, s.median_s) for s in samples])
        print(f"fitted: overhead_s={a:.2f} per_page_s={b:.3f} r_squared={r2:.3f}")
        return OverheadEstimate(
            engine=engine or "unknown", samples=samples,
            overhead_s=a, per_page_s=b, r_squared=r2,
        )

    if engine_url is not None:
        client = Client(engine_url, api_key=api_key)
        try:
            return _run(client)
        finally:
            client.close()

    with LocalEngineProcess(engine, port, ready_timeout_s=ready_timeout_s,
                             estravon_cmd=estravon_cmd) as proc:
        client = Client(proc.base_url, api_key=None)
        try:
            return _run(client)
        finally:
            client.close()

### Try it: the fit itself, with numbers we already know the answer to

`estimate_overhead()` needs a real engine to demonstrate end to end (next
cell), but `_fit_line()` is plain arithmetic and we can check it against
data we construct ourselves. Below, we build points that lie exactly on
`time = 2.0 + 0.5 * n` with no noise at all, and check that the fit recovers
`overhead_s = 2.0`, `per_page_s = 0.5`, and `r_squared = 1.0` (a perfect
line has nothing left for the model to fail to explain).

In [ ]:
#| hide
a, b, r2 = _fit_line([(1.0, 2.5), (5.0, 4.5), (10.0, 7.0), (20.0, 12.0)])
assert abs(a - 2.0) < 1e-9
assert abs(b - 0.5) < 1e-9
assert abs(r2 - 1.0) < 1e-9
print(f"overhead_s={a:.3f} per_page_s={b:.3f} r_squared={r2:.3f}")

### Try it: a real calibration run

The cell below is marked `#| eval: false` -- it makes real requests to a
real engine (small ones: three page counts, two reps each, against
`mistral`, the cheapest engine available -- roughly a few cents), so it does
not run automatically as part of `nbdev_test`. Run it directly to see a
real overhead estimate.

In [ ]:
#| eval: false
from estravon_bench.io import get_artusi

pdf = get_artusi()
est = estimate_overhead(str(pdf), page_counts=[1, 3, 6], reps=2, engine="mistral")
print(est.to_markdown_table())

Same calibration against a different engine -- the fitted line and the
per-page-count spread are worth comparing across engines, not just reading
in isolation.

In [ ]:
#| eval: false
est = estimate_overhead(str(pdf), page_counts=[1, 3, 6], reps=2, engine="datalab")
print(est.to_markdown_table())

---
This calibration is separate from the everyday `compare()` workflow
(`03_compare`) -- run it occasionally to know how much of a given engine's
`predict_time_s` is overhead, then read `compare()`'s timing column with
that in mind, especially at small page counts.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()